# 🌿 CNN: Crop Disease Image Classification in East Africa

**Author:** Jok Akech Atem Mabior | CMU Africa

**Dataset:** Synthetic 32×32 RGB leaf images simulating 4 disease classes for East African crops.

---

This notebook builds a Convolutional Neural Network (CNN) to classify crop leaf images into disease categories, with applications for early detection systems in smallholder farming communities.


## Background: Crop Disease in East Africa

### The Food Security Crisis

East Africa faces persistent threats to food security from crop diseases:

- **Maize Lethal Necrosis (MLN)**: Wiped out up to 90% of maize crops in affected fields across Kenya, Tanzania, and Uganda. Maize is the staple food for over 300 million people in sub-Saharan Africa.
- **Cassava Mosaic Disease (CMD)**: Affects cassava — the 'famine crop' relied on by 800M+ Africans. Caused by whitefly-transmitted geminiviruses.
- **Wheat Stem Rust (Ug99)**: A virulent strain first identified in Uganda that spread across East Africa, threatening wheat production in Ethiopia and Kenya.
- **Fall Armyworm**: An invasive pest species that causes up to 100% yield loss in maize and sorghum.

### The Detection Problem

Smallholder farmers (2-5 acre plots) typically:
- Lack access to plant pathologists or extension officers
- Cannot afford laboratory disease testing
- Identify diseases too late (after significant crop loss)
- Misidentify symptoms and apply incorrect — or no — treatment

### Computer Vision Solution

Smartphone-based CNN classifiers (like PlantVillage / Nuru app) allow farmers to:
1. Photograph a leaf with their phone
2. Get instant disease diagnosis
3. Receive tailored treatment recommendations
4. Alert agricultural authorities to emerging disease outbreaks

Projects like **ICIPE** (Nairobi) and **CIMMYT** are deploying similar systems across East Africa.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("TensorFlow not available — using sklearn fallback")
    tf = None

## Synthetic Leaf Image Generation

We generate 32×32 RGB images with class-specific textures and color distributions:
- **Healthy**: Uniform green with low noise
- **Bacterial Blight**: Brown/yellow with dark necrotic spots
- **Leaf Rust**: Orange/rust pustule coloration
- **Mosaic Virus**: Mottled green/yellow mosaic pattern


In [ ]:
np.random.seed(42)
IMG_SIZE = 32
N_PER_CLASS = 200
CLASSES = ['Healthy', 'Bacterial_Blight', 'Leaf_Rust', 'Mosaic_Virus']

def generate_leaf_image(cls, img_size=32):
    """Simulate leaf images with class-specific textures and colors."""
    img = np.zeros((img_size, img_size, 3), dtype=np.float32)
    if cls == 'Healthy':
        # Uniform green, low noise
        img[:,:,1] = np.random.normal(0.55, 0.06, (img_size, img_size)).clip(0.3, 0.85)
        img[:,:,0] = np.random.normal(0.12, 0.03, (img_size, img_size)).clip(0, 0.3)
        img[:,:,2] = np.random.normal(0.10, 0.03, (img_size, img_size)).clip(0, 0.3)
    elif cls == 'Bacterial_Blight':
        # Brown/yellow patches
        img[:,:,0] = np.random.normal(0.55, 0.10, (img_size, img_size)).clip(0.2, 0.9)
        img[:,:,1] = np.random.normal(0.40, 0.10, (img_size, img_size)).clip(0.1, 0.7)
        img[:,:,2] = np.random.normal(0.10, 0.05, (img_size, img_size)).clip(0, 0.3)
        # Add random dark spots
        for _ in range(np.random.randint(3, 8)):
            x, y = np.random.randint(0, img_size, 2)
            r = np.random.randint(2, 5)
            yy, xx = np.ogrid[-y:img_size-y, -x:img_size-x]
            mask = xx*xx + yy*yy <= r*r
            img[mask, 0] *= 0.4
    elif cls == 'Leaf_Rust':
        # Orange/rust colored pustules
        img[:,:,1] = np.random.normal(0.35, 0.08, (img_size, img_size)).clip(0.1, 0.6)
        img[:,:,0] = np.random.normal(0.65, 0.10, (img_size, img_size)).clip(0.3, 0.9)
        img[:,:,2] = np.random.normal(0.05, 0.02, (img_size, img_size)).clip(0, 0.2)
    else:  # Mosaic Virus
        # Mottled green/yellow pattern
        base = np.random.choice([0.25, 0.55], size=(img_size, img_size), p=[0.4, 0.6])
        img[:,:,1] = (base + np.random.normal(0, 0.06, (img_size, img_size))).clip(0.1, 0.9)
        img[:,:,0] = np.random.normal(0.20, 0.05, (img_size, img_size)).clip(0, 0.5)
        img[:,:,2] = np.random.normal(0.08, 0.03, (img_size, img_size)).clip(0, 0.3)
    return img.clip(0, 1)

# Generate dataset
images, labels = [], []
for cls_idx, cls_name in enumerate(CLASSES):
    for _ in range(N_PER_CLASS):
        images.append(generate_leaf_image(cls_name, IMG_SIZE))
        labels.append(cls_idx)

X = np.array(images, dtype=np.float32)
y = np.array(labels)

# Shuffle
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]

print(f"Image dataset: {X.shape}")
print(f"Label distribution: {np.bincount(y)}")
print(f"Classes: {CLASSES}")
print(f"Pixel value range: {X.min():.3f} — {X.max():.3f}")

## EDA 1: Sample Images by Disease Class


In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
for cls_idx, cls_name in enumerate(CLASSES):
    cls_images = X[y == cls_idx]
    for j in range(5):
        axes[cls_idx][j].imshow(cls_images[j])
        axes[cls_idx][j].axis('off')
        if j == 0:
            axes[cls_idx][j].set_ylabel(cls_name.replace('_', '\n'),
                                         rotation=0, labelpad=60, fontsize=9)
plt.suptitle('Sample Leaf Images by Disease Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## EDA 2: Mean Image per Class


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, cls in enumerate(CLASSES):
    mean_img = X[y == i].mean(axis=0)
    axes[i].imshow(mean_img)
    axes[i].set_title(f'Mean: {cls}', fontsize=10, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('Mean Image per Disease Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## EDA 3: RGB Channel Statistics per Class


In [ ]:
channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['#e74c3c', '#2ecc71', '#3498db']

stats = []
for cls_idx, cls_name in enumerate(CLASSES):
    cls_imgs = X[y == cls_idx]
    for ch_idx, ch_name in enumerate(channel_names):
        ch_vals = cls_imgs[:, :, :, ch_idx].flatten()
        stats.append({'Class': cls_name, 'Channel': ch_name,
                       'Mean': ch_vals.mean(), 'Std': ch_vals.std()})

stats_df = pd.DataFrame(stats)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ch_idx, (ch_name, ch_color) in enumerate(zip(channel_names, channel_colors)):
    ch_data = stats_df[stats_df['Channel'] == ch_name]
    axes[ch_idx].bar(ch_data['Class'], ch_data['Mean'], color=ch_color, alpha=0.75,
                     yerr=ch_data['Std'], capsize=5, edgecolor='white')
    axes[ch_idx].set_title(f'{ch_name} Channel Mean ± Std', fontweight='bold')
    axes[ch_idx].set_xlabel('Disease Class')
    axes[ch_idx].set_ylabel('Mean Pixel Value')
    axes[ch_idx].tick_params(axis='x', rotation=25)
    axes[ch_idx].set_ylim(0, 0.9)

plt.suptitle('RGB Channel Statistics by Disease Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(stats_df.round(4).to_string(index=False))

## EDA 4: Pixel Intensity Distributions


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for cls_idx, cls_name in enumerate(CLASSES):
    cls_imgs = X[y == cls_idx]
    for ch_idx, (ch_name, ch_color) in enumerate(zip(channel_names, channel_colors)):
        ch_vals = cls_imgs[:, :, :, ch_idx].flatten()
        axes[cls_idx].hist(ch_vals, bins=50, alpha=0.6, color=ch_color,
                            label=ch_name, density=True)
    axes[cls_idx].set_title(f'{cls_name} — Pixel Distribution', fontweight='bold')
    axes[cls_idx].set_xlabel('Pixel Value')
    axes[cls_idx].set_ylabel('Density')
    axes[cls_idx].legend()

plt.suptitle('Pixel Intensity Distributions by Class and Channel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Data Preprocessing and Train/Val/Test Split


In [ ]:
from tensorflow.keras.utils import to_categorical

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

y_train_cat = to_categorical(y_train, num_classes=4)
y_val_cat   = to_categorical(y_val,   num_classes=4)
y_test_cat  = to_categorical(y_test,  num_classes=4)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Train class distribution: {np.bincount(y_train)}")
print(f"Val class distribution:   {np.bincount(y_val)}")
print(f"Test class distribution:  {np.bincount(y_test)}")

## CNN Architecture

A 3-block CNN with BatchNormalization, MaxPooling, and Dropout for regularization:
- **Block 1**: 2× Conv2D(32) + BatchNorm + MaxPool + Dropout(0.25)
- **Block 2**: 2× Conv2D(64) + BatchNorm + MaxPool + Dropout(0.25)
- **Block 3**: Conv2D(128) + BatchNorm + GlobalAveragePooling
- **Head**: Dense(256) + Dropout(0.5) + Dense(4, softmax)


In [ ]:
def build_cnn(input_shape=(32, 32, 3), num_classes=4):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        # Classifier head
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_cnn()
model.summary()

## Model Training


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

## Training Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()
plt.suptitle('CNN Training History — Crop Disease Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Model Evaluation


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

y_pred = np.argmax(model.predict(X_test), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix — Crop Disease CNN', fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Misclassified Examples Visualization


In [ ]:
misclassified = np.where(y_pred != y_test)[0]
print(f"Total misclassified: {len(misclassified)} / {len(y_test)} ({len(misclassified)/len(y_test)*100:.1f}%)")

n_show = min(16, len(misclassified))
if n_show > 0:
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.flatten()
    for i in range(n_show):
        idx = misclassified[i]
        axes[i].imshow(X_test[idx])
        axes[i].set_title(
            f'True: {CLASSES[y_test[idx]]}\nPred: {CLASSES[y_pred[idx]]}',
            fontsize=7, color='red'
        )
        axes[i].axis('off')
    for j in range(n_show, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Misclassified Leaf Images', fontsize=13, fontweight='bold', color='red')
    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications found — perfect test accuracy!")

## Class Activation Mapping (CAM) — Simplified


In [ ]:
# Visualize what the CNN 'sees' by examining the last conv layer activations
# Build a model that outputs the last conv layer features
feature_model = tf.keras.Model(inputs=model.input,
                                outputs=model.layers[-6].output)  # GlobalAveragePooling input

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for cls_idx, cls_name in enumerate(CLASSES):
    # Pick one sample of this class
    sample_idx = np.where(y_test == cls_idx)[0][0]
    sample_img = X_test[sample_idx:sample_idx+1]

    # Get activation map (last conv block output: 8x8x128)
    feat_map = feature_model.predict(sample_img, verbose=0)[0]  # (8, 8, 128)
    # Average across channels for visualization
    act_map = feat_map.mean(axis=-1)

    # Original image
    axes[0][cls_idx].imshow(X_test[sample_idx])
    axes[0][cls_idx].set_title(f'{cls_name}\n(Original)', fontsize=9, fontweight='bold')
    axes[0][cls_idx].axis('off')

    # Activation map
    axes[1][cls_idx].imshow(act_map, cmap='hot', interpolation='nearest')
    axes[1][cls_idx].set_title(f'{cls_name}\n(Activation)', fontsize=9, fontweight='bold')
    axes[1][cls_idx].axis('off')

plt.suptitle('Original Images vs CNN Activation Maps (Last Conv Block)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Per-Class Confidence Distribution


In [ ]:
y_probs = model.predict(X_test, verbose=0)
max_probs = y_probs.max(axis=1)  # confidence of predicted class

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence histogram
axes[0].hist(max_probs, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(max_probs.mean(), color='red', ls='--', label=f'Mean={max_probs.mean():.3f}')
axes[0].set_title('Prediction Confidence Distribution', fontweight='bold')
axes[0].set_xlabel('Max Softmax Probability')
axes[0].set_ylabel('Count')
axes[0].legend()

# Confidence by correctness
correct_mask = y_pred == y_test
axes[1].hist(max_probs[correct_mask], bins=25, alpha=0.6, color='#2ecc71', label='Correct', density=True)
axes[1].hist(max_probs[~correct_mask], bins=25, alpha=0.6, color='#e74c3c', label='Incorrect', density=True)
axes[1].set_title('Confidence: Correct vs Incorrect Predictions', fontweight='bold')
axes[1].set_xlabel('Max Softmax Probability')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Model Confidence Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Avg confidence (correct):   {max_probs[correct_mask].mean():.4f}")
print(f"Avg confidence (incorrect): {max_probs[~correct_mask].mean():.4f}")

## Conclusions

### CNN Performance

The 3-block CNN with BatchNormalization and GlobalAveragePooling achieved strong performance on synthetic leaf image classification:

| Class | Key Visual Feature | Typical Performance |
|-------|-------------------|--------------------|
| Healthy | Uniform green texture | High precision/recall |
| Bacterial Blight | Brown patches + dark spots | Moderate — confusion with Leaf Rust |
| Leaf Rust | Orange/rust coloration | Good separation |
| Mosaic Virus | Mottled green/yellow | Moderate — complex pattern |

### Architecture Insights
- **BatchNormalization** stabilized training and improved convergence speed
- **GlobalAveragePooling** reduced parameter count vs Flatten, reducing overfitting
- **EarlyStopping** prevented overfitting on the small (160 training samples per class) dataset

### Real-World Applications

1. **Farmer-Facing Mobile Apps**: Integrate into apps like Nuru (ICIPE) or Digital Green — farmers photograph leaves with USD 30 smartphones
2. **Offline Deployment**: TensorFlow Lite model (< 5MB) can run without internet on Android, critical in rural East Africa
3. **Drone-Based Monitoring**: CNN classifies aerial imagery to detect disease spread at field scale
4. **Early Warning Systems**: Aggregate reports to national databases to trigger disease alerts

### Limitations of Synthetic Data
- Real leaf images have complex backgrounds, varying lighting, and partial occlusion
- Intra-class variation in real diseases is much higher than simulated
- Model trained on synthetic data will not transfer directly to field images

### Next Steps
- **Transfer Learning**: Fine-tune MobileNetV2 or EfficientNet-B0 on real PlantVillage dataset (~54,000 images)
- **Data Augmentation**: Albumentations library — flips, rotations, brightness, cutout
- **Federated Learning**: Train across farmers' phones without centralizing photos (privacy-preserving)
- **Multi-task Learning**: Simultaneously predict disease type AND severity level
